# B1 — TF-IDF + Logistic Regression

## Imports and data loading

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, mean_absolute_error

SEED = 42
VAL_SIZE = 0.10

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

## Canonical 90/10 train/val split

Stratified on `label`, fixed seed. Indices saved to disk so every later model uses the exact same split.

In [2]:
val_path = 'data/val_indices.npy'
if os.path.exists(val_path):
    val_idx = np.load(val_path)
else:
    _, val_idx = train_test_split(
        np.arange(len(train)), test_size=VAL_SIZE,
        stratify=train['label'], random_state=SEED,
    )
    np.save(val_path, val_idx)

train_df = train.drop(index=val_idx).reset_index(drop=True)
val_df   = train.loc[val_idx].reset_index(drop=True)

## TF-IDF vectorization

Unigrams + bigrams, sublinear TF, 50k-feature cap. Fit on train only.

In [3]:
vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=2,
    sublinear_tf=True,
)
X_train = vec.fit_transform(train_df['sentence'])
X_val   = vec.transform(val_df['sentence'])
X_test  = vec.transform(test['sentence'])
y_train = train_df['label'].values
y_val   = val_df['label'].values

## Train logistic regression

Multinomial LogReg on the TF-IDF features.

In [4]:
clf = LogisticRegression(C=1.0, max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)

/Users/ismaillataoui/CompIntelLab/Project/code/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=200, n_jobs=-1)

## Evaluate — train and val

Reports score, MAE, accuracy, confusion matrix. Comparing train vs val tells us how much we're overfitting.

In [5]:
def report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    score = 1 - mae / 4
    acc = float(np.mean(y_true == y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    print(f'[{name}] score={score:.4f}  MAE={mae:.4f}  acc={acc:.4f}')
    print(cm)
    return score

report(y_train, clf.predict(X_train), 'train')
report(y_val,   clf.predict(X_val),   'val')

[train] score=0.9130  MAE=0.3479  acc=0.7221
[[36057  6504  2148   384   267]
 [ 6648 30107  6748  1409   448]
 [ 2910  6937 29294  4872  1347]
 [  655  1614  5441 29453  8197]
 [  309   413   909  4878 38851]]
[val] score=0.8761  MAE=0.4958  acc=0.5944
[[3533 1111  294   60   42]
 [1134 2472 1184  180   70]
 [ 408 1199 2410  868  155]
 [  92  223  873 2659 1193]
 [  62   62  133  877 3906]]


0.8760615079365079

## MAE-optimal rounding

`argmax` picks the mode of the predictive distribution. For an ordinal task with MAE loss, the **median** is the optimal point estimate. Same model, smarter rounding → free improvement.

In [6]:
def median_round(probs):
    cum = np.cumsum(probs, axis=1)
    return np.argmax(cum >= 0.5, axis=1)

val_probs = clf.predict_proba(X_val)
report(y_val, median_round(val_probs), 'val_med')

[val_med] score=0.8822  MAE=0.4713  acc=0.5896
[[2922 1714  351   47    6]
 [ 640 2896 1351  136   17]
 [ 167 1294 2783  737   59]
 [  31  194 1125 2791  899]
 [  17   62  244 1252 3465]]


0.882172619047619

## Write Kaggle submission

Predict on the test set with the same MAE-optimal rounding, save `submissions/b1_tfidf_logreg.csv`.

In [7]:
os.makedirs('submissions', exist_ok=True)
test_pred = median_round(clf.predict_proba(X_test))
pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)}).to_csv(
    'submissions/b1_tfidf_logreg.csv', index=False,
)

## Top features per class

Top-10 ngrams pushing toward each rating. Useful figure for the report.

In [8]:
names = vec.get_feature_names_out()
for cls in range(5):
    top = np.argsort(clf.coef_[cls])[-10:][::-1]
    print(cls, [names[i] for i in top])

0 ['one star', 'nicht', 'stern', 'schrott', 'not', 'unbrauchbar', 'horrible', 'terrible', 'nie', 'garbage']
1 ['two stars', 'zwei sterne', 'not', 'nicht', 'leider', 'disappointed', 'schade', 'disappointing', 'naja', 'enttäuscht']
2 ['three stars', 'drei sterne', 'but', 'sterne abzug', 'leider', 'ok', 'okay', 'aber', 'however', 'three']
3 ['four stars', 'gut', 'stern abzug', 'good', 'abzug', 'not bad', 'great', 'etwas', 'four', 'gute']
4 ['great', 'super', 'five stars', 'perfect', 'love', 'perfekt', 'five', 'top', 'excellent', 'awesome']
